# LeetCode 391 — Perfect Rectangle bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def isRectangleCover(self, rectangles: List[List[int]]) -> bool:

```

Tên bài dịch nôm na là **"Hình chữ nhật hoàn hảo"**: cho một đống hình chữ nhật nhỏ, hỏi chúng có **ghép khít** lại thành **đúng một** hình chữ nhật lớn hay không — giống như kiểm tra một bức tranh ghép (puzzle) hay một nền nhà lát gạch: **không hở chỗ nào** và **không có viên gạch nào chồng lên viên khác**.

Mục tiêu của bài học:

1. Biết cách **biểu diễn** hình chữ nhật bằng 4 số `[x, y, a, b]`, tính **diện tích**, tìm **hình bao** (bounding box) bằng `min` / `max`, và tự **vẽ ASCII** các hình để "nhìn thấy" bài toán.
2. Tự nghĩ ra cách **trực giác nhất**: tô từng ô vuông đơn vị vào một `set` (cách 1) — rồi hiểu vì sao nó chết khi tọa độ lên tới `10^5`.
3. Học cách kiểm tra **hai đoạn / hai hình chữ nhật có chồng nhau không**, từ đó viết cách 2: **kiểm mọi cặp** `O(n²)` + **so diện tích** — và thấy nó vẫn quá chậm với `n = 2·10^4`.
4. Nắm **insight cốt lõi** của lời giải chính `O(n)`: **diện tích khớp** + **tập góc "bật/tắt"** (toggle, giống XOR) chỉ còn đúng 4 góc của hình bao. Hiểu **vì sao** điều kiện này *cần* (mỗi điểm bên trong là góc của số chẵn hình) và *đủ* (bổ đề "đếm góc nằm dưới-trái" → mọi điểm được phủ số lẻ lần → không hở; diện tích → không chồng). Có **phản ví dụ** cho thấy chỉ kiểm diện tích, hoặc chỉ kiểm góc, đều **sai**.
5. (Nâng cao) Học kỹ thuật **sweep line** (quét đường thẳng) với `bisect` để phát hiện chồng lấn trong `O(n log n)`-ish.
6. Đối chiếu chéo tất cả các cách với brute force trên hàng nghìn input ngẫu nhiên, tránh các lỗi kinh điển (quên kiểm diện tích, chỉ kiểm "4 góc có trong set", dùng `list` làm phần tử `set`, coi "chạm cạnh" là chồng…) và nhận dạng pattern cho các bài hình học lưới khác.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho mảng `rectangles`, trong đó `rectangles[i] = [x_i, y_i, a_i, b_i]` biểu diễn một hình chữ nhật **có cạnh song song với trục tọa độ** (axis-aligned). Điểm **dưới-trái** của hình là `(x_i, y_i)` và điểm **trên-phải** là `(a_i, b_i)`.
>
> Trả về `true` nếu **tất cả** các hình chữ nhật **cùng nhau tạo thành một phủ chính xác** (exact cover) của **một vùng hình chữ nhật**.

**Example 1**

```text
Input: rectangles = [[1,1,3,3],[3,1,4,2],[3,2,4,4],[1,3,2,4],[2,3,3,4]]
Output: true
Explanation: All 5 rectangles together form an exact cover of a rectangular region.
```

**Example 2**

```text
Input: rectangles = [[1,1,2,3],[1,3,2,4],[3,1,4,2],[3,2,4,4]]
Output: false
Explanation: Because there is a gap between the two rectangular regions.
```

**Example 3**

```text
Input: rectangles = [[1,1,3,3],[3,1,4,2],[1,3,2,4],[2,2,4,4]]
Output: false
Explanation: Because two of the rectangles overlap with each other.
```

**Constraints**

- `1 <= rectangles.length <= 2 * 10^4`
- `rectangles[i].length == 4`
- `-10^5 <= x_i < a_i <= 10^5`
- `-10^5 <= y_i < b_i <= 10^5`

### 1.1 "Phủ chính xác" nghĩa là gì?

Gọi `n` là số hình chữ nhật. Ta trả `True` khi và chỉ khi **cả ba** điều sau cùng đúng:

1. **Hợp lại thành một hình chữ nhật**: phần mặt phẳng mà các hình chiếm, ghép lại, phải là đúng một hình chữ nhật (không phải hình chữ L, không phải hai mảnh rời).
2. **Không hở (no gap)**: mọi điểm bên trong hình chữ nhật lớn đó đều thuộc ít nhất một hình nhỏ.
3. **Không chồng (no overlap)**: không có phần diện tích nào bị **hai** hình nhỏ cùng phủ.

Một vài điều người mới hay băn khoăn:

- **Chạm cạnh / chạm góc không tính là chồng.** Hình `[1,1,3,3]` và `[3,1,4,2]` dùng chung đoạn thẳng `x = 3` — đó là "ghép sát", hoàn toàn hợp lệ. Chồng nghĩa là có **phần diện tích dương** chung.
- **Tọa độ là các điểm lưới**, nhưng hình chữ nhật là **vùng liên tục**. Hình `[1,1,3,3]` có chiều rộng `3 − 1 = 2`, chiều cao `3 − 1 = 2`, diện tích `4`.
- `x_i < a_i` và `y_i < b_i` luôn đúng, nên **không có hình nào suy biến** (diện tích 0).
- `n = 1` thì luôn `True`: một hình chữ nhật tự phủ chính nó.

Cell dưới khai báo các ví dụ dùng suốt bài.

In [1]:
from typing import List
from collections import Counter
import bisect
import random
import time

ex1 = [[1, 1, 3, 3], [3, 1, 4, 2], [3, 2, 4, 4], [1, 3, 2, 4], [2, 3, 3, 4]]  # true
ex2 = [[1, 1, 2, 3], [1, 3, 2, 4], [3, 1, 4, 2], [3, 2, 4, 4]]                # false (hở)
ex3 = [[1, 1, 3, 3], [3, 1, 4, 2], [1, 3, 2, 4], [2, 2, 4, 4]]                # false (chồng)

for name, rs in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3)]:
    print(f"{name}: n = {len(rs)} hình -> {rs}")

ex1: n = 5 hình -> [[1, 1, 3, 3], [3, 1, 4, 2], [3, 2, 4, 4], [1, 3, 2, 4], [2, 3, 3, 4]]
ex2: n = 4 hình -> [[1, 1, 2, 3], [1, 3, 2, 4], [3, 1, 4, 2], [3, 2, 4, 4]]
ex3: n = 4 hình -> [[1, 1, 3, 3], [3, 1, 4, 2], [1, 3, 2, 4], [2, 2, 4, 4]]


## 2. Kiến thức nền 1 — Biểu diễn một hình chữ nhật

Một hình chữ nhật `[x, y, a, b]`:

```text
 y=b  (x,b) +-----------+ (a,b)      <- góc trên-trái, trên-phải
            |           |
            |           |
 y=y  (x,y) +-----------+ (a,y)      <- góc dưới-trái, dưới-phải
           x=x         x=a
```

- **Chiều rộng** `= a − x`, **chiều cao** `= b − y`, **diện tích** `= (a − x) · (b − y)`.
- **4 góc**: `(x, y)`, `(a, y)`, `(x, b)`, `(a, b)`. Bốn góc này sẽ là "nhân vật chính" của lời giải tối ưu.

Trong Python ta "bóc" (unpack) 4 số ra 4 biến chỉ bằng một dòng: `x, y, a, b = r`. Ví dụ `r = [1, 1, 3, 3]` thì `x = 1, y = 1, a = 3, b = 3`.

In [2]:
def area(r):
    x, y, a, b = r                      # unpack: bóc 4 số ra 4 biến
    return (a - x) * (b - y)

def corners(r):
    x, y, a, b = r
    return [(x, y), (a, y), (x, b), (a, b)]

r = [1, 1, 3, 3]
print("rộng =", r[2] - r[0], "| cao =", r[3] - r[1], "| diện tích =", area(r))
print("4 góc:", corners(r))
print("diện tích từng hình của ex1:", [area(r) for r in ex1], "-> tổng", sum(area(r) for r in ex1))

rộng = 2 | cao = 2 | diện tích = 4
4 góc: [(1, 1), (3, 1), (1, 3), (3, 3)]
diện tích từng hình của ex1: [4, 1, 2, 1, 1] -> tổng 9


## 3. Kiến thức nền 2 — Hình bao (bounding box)

Nếu các hình nhỏ ghép khít thành **một** hình chữ nhật lớn, thì hình lớn đó **bắt buộc** phải là **hình bao nhỏ nhất** chứa tất cả các hình nhỏ:

- cạnh trái `X = min(x_i)`, cạnh dưới `Y = min(y_i)`,
- cạnh phải `A = max(a_i)`, cạnh trên `B = max(b_i)`.

Tại sao? Hình lớn phải chứa mọi hình nhỏ, nên nó phải lan ít nhất tới `min x_i` bên trái, `max a_i` bên phải…; nhưng nó cũng **không được** lan xa hơn, vì phần lan thêm không có hình nhỏ nào phủ (sẽ bị hở). Vậy **ứng viên duy nhất** là hình bao `[X, Y, A, B]` — ta không phải đoán hình lớn ở đâu.

Ví dụ ex1: `min x = 1, min y = 1, max a = 4, max b = 4` → hình bao `[1, 1, 4, 4]`, diện tích `3 · 3 = 9`.

In [3]:
def bounding_box(rects):
    X = min(r[0] for r in rects)
    Y = min(r[1] for r in rects)
    A = max(r[2] for r in rects)
    B = max(r[3] for r in rects)
    return [X, Y, A, B]

for name, rs in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3)]:
    box = bounding_box(rs)
    print(f"{name}: hình bao = {box}, diện tích bao = {area(box)}, tổng diện tích các hình = {sum(area(r) for r in rs)}")

ex1: hình bao = [1, 1, 4, 4], diện tích bao = 9, tổng diện tích các hình = 9
ex2: hình bao = [1, 1, 4, 4], diện tích bao = 9, tổng diện tích các hình = 6
ex3: hình bao = [1, 1, 4, 4], diện tích bao = 9, tổng diện tích các hình = 10


## 4. Kiến thức nền 3 — Vẽ ASCII để nhìn thấy bài toán

Hình học mà không vẽ thì rất dễ "lú". Ta viết một hàm `draw(rects)` vẽ các hình lên lưới ký tự:

- Chia hình bao thành các **ô vuông đơn vị** (unit cell). Ô được ghi nhãn `(i, j)` là hình vuông `[i, i+1] × [j, j+1]`, tức là có góc dưới-trái tại `(i, j)`.
- Ô `(i, j)` nằm trong hình `[x, y, a, b]` khi `x <= i < a` và `y <= j < b` (chú ý dấu `<` ở bên phải: ô bắt đầu tại `i = a` đã nằm **ngoài** hình).
- Hình thứ `k` được đặt tên bằng chữ cái `A, B, C, …` (`chr(65 + k)`).
- Ký hiệu trên lưới: **chữ cái** = ô được phủ đúng 1 lần (bởi hình đó), **`.`** = ô trống (**hở**), **chữ số `2`, `3`…** = ô bị phủ chồng bấy nhiêu lớp (**chồng**).
- Trục `y` hướng **lên trên** như toán học, nên ta in hàng `y` lớn nhất trước.

Chỉ dùng hàm này cho tọa độ nhỏ (0–9) để lưới gọn gàng.

In [4]:
def draw(rects, title=None):
    X, Y, A, B = bounding_box(rects)
    if title:
        print(title)
    for j in range(B - 1, Y - 1, -1):               # in từ hàng trên cùng xuống
        row = []
        for i in range(X, A):
            owners = [chr(65 + k) for k, (x, y, a, b) in enumerate(rects)
                      if x <= i < a and y <= j < b]
            if not owners:
                row.append(".")                      # hở
            elif len(owners) == 1:
                row.append(owners[0])                # phủ đúng 1 lần
            else:
                row.append(str(len(owners)))         # chồng nhiều lớp
        print(f"  y={j:>2} | " + " ".join(row))
    print("         +" + "--" * (A - X))
    print("      x =   " + " ".join(str(i) for i in range(X, A)))

draw([[0, 0, 3, 2]], "Một hình [0,0,3,2]: rộng 3, cao 2")

Một hình [0,0,3,2]: rộng 3, cao 2
  y= 1 | A A A
  y= 0 | A A A
         +------
      x =   0 1 2


In [5]:
draw(ex1, "ex1 (true):")
print()
draw(ex2, "ex2 (false - hở):")
print()
draw(ex3, "ex3 (false - chồng):")

ex1 (true):
  y= 3 | D E C
  y= 2 | A A C
  y= 1 | A A B
         +------
      x =   1 2 3

ex2 (false - hở):
  y= 3 | B . D
  y= 2 | A . D
  y= 1 | A . C
         +------
      x =   1 2 3

ex3 (false - chồng):
  y= 3 | C D D
  y= 2 | A 2 D
  y= 1 | A A B
         +------
      x =   1 2 3


Đọc hình:

- **ex1**: lưới `3 × 3` kín, không có `.` và không có chữ số → phủ hoàn hảo. Hình `A = [1,1,3,3]` chiếm khối `2 × 2` dưới-trái; `B, C` là cột bên phải; `D, E` là hàng trên.
- **ex2**: cột giữa `x = 2` toàn `.` → có một **khe hở** dọc giữa hai cột hình.
- **ex3**: ô `(2, 2)` hiện số `2` → hình `A = [1,1,3,3]` và `D = [2,2,4,4]` **chồng** lên nhau ở ô đó.

Nhìn hình thì người thấy ngay. Câu hỏi của bài là: **máy kiểm tra điều đó nhanh thế nào**, khi có tới `2·10^4` hình và tọa độ tới `±10^5`?

## 5. Kiến thức nền 4 — `tuple`, `set` và `Counter` cho điểm trên mặt phẳng

Ta sẽ cần lưu **các điểm** `(x, y)` vào `set` (tập hợp) hoặc `Counter` (bộ đếm).

- `set` là "túi" chứa các phần tử **không trùng**, kiểm tra `p in s` nhanh `O(1)` trung bình (nhờ bảng băm — hash table).
- Phần tử của `set` / key của `dict` phải **hashable** (băm được, tức là không đổi được). `tuple` `(1, 2)` thì được; `list` `[1, 2]` thì **không** (list có thể sửa nên Python cấm băm). Vì vậy góc luôn được viết dạng `(x, y)`.
- `Counter` là `dict` đặc biệt: `cnt[p] += 1` đếm số lần gặp `p`, chưa có thì mặc định là `0`.
- `s.add(p)` thêm, `s.remove(p)` xóa (báo lỗi nếu không có), `s.discard(p)` xóa (không báo lỗi).

In [6]:
s = set()
s.add((1, 2))
s.add((3, 4))
s.add((1, 2))                 # thêm trùng: không có tác dụng
print("set:", s, "| (1, 2) in s ->", (1, 2) in s, "| (2, 1) in s ->", (2, 1) in s)

cnt = Counter()
for p in [(1, 2), (3, 4), (1, 2), (1, 2)]:
    cnt[p] += 1
print("Counter:", dict(cnt), "| cnt[(9, 9)] =", cnt[(9, 9)])

try:
    s.add([5, 6])             # list không hashable
except TypeError as e:
    print("Dùng list làm phần tử set -> TypeError:", e)

set: {(1, 2), (3, 4)} | (1, 2) in s -> True | (2, 1) in s -> False
Counter: {(1, 2): 3, (3, 4): 1} | cnt[(9, 9)] = 0
Dùng list làm phần tử set -> TypeError: unhashable type: 'list'


## 6. Cách 1 — Brute force: tô từng ô đơn vị

Đây là cách một người mới sẽ tự nghĩ ra, đúng như cách ta "nhìn hình" ở trên:

1. Chuẩn bị một `set` tên `painted` chứa các ô đã được tô.
2. Với mỗi hình `[x, y, a, b]`, duyệt mọi ô `(i, j)` với `x <= i < a`, `y <= j < b`:
   - Nếu ô đã có trong `painted` → ô này bị **hai** hình phủ → **chồng** → trả `False` ngay.
   - Nếu chưa → tô nó (`painted.add((i, j))`).
3. Hết vòng lặp mà không chồng: mỗi ô được tô đúng 1 lần, và mọi ô tô đều nằm trong hình bao (vì hình bao chứa mọi hình). Hình bao có `area(box)` ô. Vậy **không hở** ⇔ `len(painted) == area(box)`.

Vì sao chỉ cần đếm? Các ô tô là một **tập con** của các ô trong hình bao; tập con có cùng số phần tử với tập lớn thì **bằng** tập lớn. Đó chính là "không thiếu ô nào".

In [7]:
def is_cover_cells(rects):
    painted = set()
    for x, y, a, b in rects:
        for i in range(x, a):
            for j in range(y, b):
                if (i, j) in painted:        # ô đã bị hình khác tô -> chồng
                    return False
                painted.add((i, j))
    return len(painted) == area(bounding_box(rects))   # đủ ô -> không hở

for name, rs in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3)]:
    print(name, "->", is_cover_cells(rs))

ex1 -> True
ex2 -> False
ex3 -> False


### 6.1 Cách 1 vướng ở đâu?

Thời gian và bộ nhớ của cách 1 tỉ lệ với **tổng diện tích** `S = Σ area(r)` — mỗi ô đơn vị tốn một lần `add`. Với constraints `-10^5 <= x < a <= 10^5`, chỉ **một** hình `[-10^5, -10^5, 10^5, 10^5]` đã có diện tích `(2·10^5)² = 4·10^10` ô. Python làm cỡ `10^7` thao tác set mỗi giây → mất **hơn một giờ**, chưa kể cần hàng trăm GB RAM để chứa `set`.

Bài học: cách 1 **đúng** và cực kỳ dễ tin tưởng, nên ta sẽ giữ nó làm **"trọng tài"** (oracle) để kiểm tra các cách nhanh hơn trên tọa độ nhỏ. Nhưng để nộp bài, ta phải thoát khỏi việc làm việc theo **từng ô**.

In [8]:
huge = [[-10**5, -10**5, 10**5, 10**5]]
S = area(huge[0])
print("Số ô đơn vị của một hình lớn nhất:", f"{S:,}")
print("Ước lượng thời gian với ~1e7 thao tác/giây:", round(S / 1e7 / 3600, 1), "giờ")

Số ô đơn vị của một hình lớn nhất: 40,000,000,000
Ước lượng thời gian với ~1e7 thao tác/giây: 1.1 giờ


## 7. Kiến thức nền 5 — Hai đoạn thẳng có chồng nhau không?

Trước khi làm với hình chữ nhật, hãy làm với **đoạn thẳng** trên trục số (1 chiều).

Đoạn `[l1, r1]` và `[l2, r2]` (với `l < r`) có **phần chung độ dài dương** khi và chỉ khi:

```text
max(l1, l2) < min(r1, r2)
```

Lý do: phần chung (nếu có) bắt đầu ở điểm "trái muộn nhất" `max(l1, l2)` và kết thúc ở điểm "phải sớm nhất" `min(r1, r2)`. Có phần chung thật sự khi điểm bắt đầu nằm **trước hẳn** điểm kết thúc.

| Đoạn 1 | Đoạn 2 | `max(l)` | `min(r)` | Chồng? |
|---|---|:---:|:---:|:---:|
| `[1, 4]` | `[3, 6]` | 3 | 4 | ✅ (`3 < 4`, phần chung `[3,4]`) |
| `[1, 3]` | `[3, 6]` | 3 | 3 | ❌ (`3 < 3` sai — chỉ **chạm** tại điểm 3) |
| `[1, 2]` | `[4, 6]` | 4 | 2 | ❌ (cách xa nhau) |
| `[1, 9]` | `[3, 4]` | 3 | 4 | ✅ (đoạn này chứa đoạn kia) |

Chú ý dấu `<` **nghiêm ngặt**: dùng `<=` sẽ coi "chạm" là "chồng" — một lỗi rất hay gặp.

In [9]:
def intervals_overlap(l1, r1, l2, r2):
    return max(l1, l2) < min(r1, r2)

for seg1, seg2 in [((1, 4), (3, 6)), ((1, 3), (3, 6)), ((1, 2), (4, 6)), ((1, 9), (3, 4))]:
    print(seg1, seg2, "->", intervals_overlap(*seg1, *seg2))

(1, 4) (3, 6) -> True
(1, 3) (3, 6) -> False
(1, 2) (4, 6) -> False
(1, 9) (3, 4) -> True


### 7.1 Hai hình chữ nhật có chồng nhau không?

Hai hình chữ nhật song song trục có **phần chung diện tích dương** khi và chỉ khi **hình chiếu** của chúng lên trục `x` chồng nhau **và** hình chiếu lên trục `y` cũng chồng nhau. Phần chung chính là hình chữ nhật `[max x, max y, min a, min b]`.

```text
   chồng (x và y đều chồng)      chạm cạnh (x chỉ chạm)       chạm góc
   +-----+                       +---+---+                     +---+
   |  +--+--+                    |   |   |                     |   |
   +--+--+  |                    +---+---+                     +---+---+
      +-----+                                                      |   |
                                                                   +---+
```

Chỉ cần **một** trong hai trục không chồng (chỉ chạm hoặc cách xa) là hai hình không chồng.

In [10]:
def rects_overlap(r1, r2):
    return (max(r1[0], r2[0]) < min(r1[2], r2[2]) and
            max(r1[1], r2[1]) < min(r1[3], r2[3]))

tests = [
    ("chồng (ex3: A và D)", [1, 1, 3, 3], [2, 2, 4, 4]),
    ("chạm cạnh (ex1: A và B)", [1, 1, 3, 3], [3, 1, 4, 2]),
    ("chạm góc", [0, 0, 1, 1], [1, 1, 2, 2]),
    ("chứa trọn", [0, 0, 5, 5], [1, 1, 2, 2]),
    ("cách xa", [0, 0, 1, 1], [3, 3, 4, 4]),
]
for name, r1, r2 in tests:
    print(f"{name:25s} -> {rects_overlap(r1, r2)}")

chồng (ex3: A và D)       -> True
chạm cạnh (ex1: A và B)   -> False
chạm góc                  -> False
chứa trọn                 -> True
cách xa                   -> False


## 8. Cách 2 — Kiểm mọi cặp chồng nhau + so diện tích

Mấu chốt cải tiến so với cách 1: **không cần nhìn từng ô** nữa. Ta tách "phủ chính xác" thành hai câu hỏi trả lời được bằng các con số:

- **Không chồng**: không có cặp `(i, j)` nào mà `rects_overlap(rects[i], rects[j])`. Kiểm bằng hai vòng lặp lồng nhau.
- **Không hở**: nếu các hình đã không chồng nhau, thì diện tích phần chúng chiếm **bằng đúng** tổng diện tích `Σ area(r)`. Phần đó nằm trong hình bao. Vậy **không hở** ⇔ `Σ area(r) == area(box)`.

Ví dụ số: ex2 không có cặp nào chồng, nhưng `Σ area = 2 + 1 + 1 + 2 = 6 < 9 = area(box)` → thiếu `3` đơn vị diện tích → hở (đúng là cột giữa có 3 ô trống).

Đây là **cùng một lập luận** với cách 1 (tập con có cùng "kích thước" với tập lớn thì bằng tập lớn), chỉ thay "số ô" bằng "diện tích".

In [11]:
def is_cover_pairwise(rects):
    n = len(rects)
    for i in range(n):
        for j in range(i + 1, n):                 # mỗi cặp xét đúng 1 lần
            if rects_overlap(rects[i], rects[j]):
                return False                      # chồng
    total = sum(area(r) for r in rects)
    return total == area(bounding_box(rects))     # không chồng: diện tích khớp <=> không hở

for name, rs in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3)]:
    print(name, "->", is_cover_pairwise(rs))

ex1 -> True
ex2 -> False
ex3 -> False


### 8.1 Cách 2 vướng ở đâu?

Thời gian không còn phụ thuộc vào tọa độ, nhưng số cặp là `n(n − 1)/2`. Với `n = 2·10^4` đó là khoảng **`2·10^8` cặp** — Python mất cỡ **vài chục giây cho một test duy nhất** (cell dưới đo thực tế), trong khi LeetCode chạy cả bộ nhiều test trong giới hạn vài giây → chắc chắn **Time Limit Exceeded**.

Cell dưới đo thời gian trên một lưới `n = 1000` ô vuông ghép khít (mọi cặp đều phải xét hết vì không có cặp nào chồng), rồi ngoại suy lên `n = 2·10^4` (gấp 20 lần `n` → gấp khoảng 400 lần số cặp).

Câu hỏi then chốt cho bước tiếp theo: **có cách nào phát hiện chồng/hở mà không cần so từng cặp không?** Câu trả lời nằm ở **các góc**.

In [12]:
def grid_cover(w, h, x0=0, y0=0):
    # w*h ô vuông đơn vị ghép khít thành hình [x0, y0, x0+w, y0+h]
    return [[x0 + i, y0 + j, x0 + i + 1, y0 + j + 1] for i in range(w) for j in range(h)]

rs = grid_cover(25, 40)                  # n = 1000
t0 = time.perf_counter()
ok = is_cover_pairwise(rs)
t1 = time.perf_counter()
print(f"n = {len(rs)}: kết quả {ok}, mất {t1 - t0:.2f} giây")
print(f"ngoại suy n = 20000: khoảng {(t1 - t0) * 400:.0f} giây")

n = 1000: kết quả True, mất 0.09 giây
ngoại suy n = 20000: khoảng 34 giây


## 9. Insight — Quan sát các góc

Quay lại ex1 và **đếm** xem mỗi điểm lưới là góc của bao nhiêu hình. Mỗi hình đóng góp 4 góc, 5 hình → 20 lần "góc", rơi vào 12 điểm khác nhau:

```text
  y=4   (1,4)1 ---- (2,4)2 ---- (3,4)2 ---- (4,4)1
          |    D      |     E     |           |
  y=3   (1,3)2 ---- (2,3)2 ---- (3,3)2      C |
          |                       |           |
  y=2     |          A            |   (3,2)2 -(4,2)2
          |                       |     B     |
  y=1   (1,1)1 ---------------- (3,1)2 ---- (4,1)1
```

(con số sau mỗi điểm = số hình nhận điểm đó làm góc)

Quan sát:

- 4 góc của **hình bao** `(1,1), (4,1), (1,4), (4,4)` xuất hiện **đúng 1 lần**.
- Mọi điểm khác xuất hiện **2 lần** (trong các phủ hoàn hảo khác, có thể là **4 lần** — ví dụ 4 ô vuông chụm lại ở một điểm).

Không có điểm nào xuất hiện **lẻ** lần ngoài 4 góc bao. Ta kiểm chứng bằng `Counter`.

In [13]:
def corner_counts(rects):
    cnt = Counter()
    for r in rects:
        for p in corners(r):
            cnt[p] += 1
    return cnt

cnt = corner_counts(ex1)
for p in sorted(cnt, key=lambda q: (-q[1], q[0])):          # in từ hàng trên xuống
    print(p, "->", cnt[p], "lần")

print("Phủ 2x2 bằng 4 ô vuông, điểm giữa (1,1) ->", corner_counts(grid_cover(2, 2))[(1, 1)], "lần")

(1, 4) -> 1 lần
(2, 4) -> 2 lần
(3, 4) -> 2 lần
(4, 4) -> 1 lần
(1, 3) -> 2 lần
(2, 3) -> 2 lần
(3, 3) -> 2 lần
(3, 2) -> 2 lần
(4, 2) -> 2 lần
(1, 1) -> 1 lần
(3, 1) -> 2 lần
(4, 1) -> 1 lần
Phủ 2x2 bằng 4 ô vuông, điểm giữa (1,1) -> 4 lần


### 9.1 Vì sao trong một phủ hoàn hảo, mỗi điểm (trừ 4 góc bao) là góc của số **chẵn** hình?

Lấy một điểm `p` bất kỳ và nhìn vào một hình tròn **rất nhỏ** quanh `p`. Hai đường thẳng ngang/dọc qua `p` chia hình tròn đó thành **4 góc phần tư** (quadrant):

```text
          |
    TL    |    TR
  --------p--------
    BL    |    BR
          |
```

Một hình chữ nhật chứa `p` (kể cả trên biên) sẽ phủ bao nhiêu góc phần tư quanh `p`?

| Vị trí của `p` đối với hình | Số góc phần tư hình đó phủ |
|---|:---:|
| `p` là **góc** của hình | **1** |
| `p` nằm trên **cạnh** (không phải góc) | **2** |
| `p` nằm **hẳn bên trong** | **4** |

Trong một phủ hoàn hảo, mỗi góc phần tư **nằm trong hình bao** được phủ **đúng 1 lần** (không hở, không chồng). Gọi `c` = số hình nhận `p` làm góc, `e` = số hình có `p` trên cạnh, `i` = số hình chứa `p` bên trong. Đếm tổng số "lượt phủ góc phần tư":

- `p` **bên trong** hình bao: 4 góc phần tư đều được phủ đúng 1 lần → `c + 2e + 4i = 4` → `c = 4 − 2e − 4i` → **chẵn** (`0`, `2` hoặc `4`).
- `p` trên **cạnh** hình bao (không phải góc): chỉ 2 góc phần tư nằm trong → `c + 2e = 2` → `c ∈ {0, 2}`, **chẵn**.
- `p` là **góc** hình bao: chỉ 1 góc phần tư nằm trong → `c = 1`, **lẻ**.

Kết luận (điều kiện **cần**): phủ hoàn hảo ⇒ **những điểm là góc của số lẻ hình chính là đúng 4 góc của hình bao**. Và tất nhiên còn ⇒ `Σ area = area(box)`.

### 9.2 Kiến thức nền 6 — "Bật/tắt" (toggle) một phần tử trong `set` = đếm chẵn lẻ

Ta chỉ quan tâm **chẵn hay lẻ**, không cần con số chính xác. Mẹo: dùng `set` như một dãy **công tắc đèn**:

- gặp điểm `p` lần đầu → **bật** (thêm vào set),
- gặp lại → **tắt** (xóa khỏi set),
- gặp lần ba → bật lại, …

Cuối cùng, `p` nằm trong set ⇔ `p` được gặp **số lẻ** lần. Đây chính là phép **XOR** (`^`, "hoặc loại trừ"): `0 ^ 1 = 1`, `1 ^ 1 = 0`. Với set, Python có sẵn toán tử **hiệu đối xứng** `s ^= {p}` làm đúng việc bật/tắt này.

Ví dụ: dãy `(1,1), (2,2), (1,1), (1,1)` → `(1,1)` gặp 3 lần (lẻ) → còn; `(2,2)` gặp 1 lần → còn. Nếu thêm một `(2,2)` nữa → `(2,2)` gặp 2 lần → tắt.

In [14]:
def toggle_set(points):
    s = set()
    for p in points:
        if p in s:
            s.remove(p)     # đang bật -> tắt
        else:
            s.add(p)        # đang tắt -> bật
    return s

pts = [(1, 1), (2, 2), (1, 1), (1, 1), (2, 2), (3, 3)]
s2 = set()
for p in pts:
    s2 ^= {p}               # cách viết gọn bằng hiệu đối xứng

odd_by_counter = {p for p, c in Counter(pts).items() if c % 2 == 1}
print("toggle    :", toggle_set(pts))
print("s ^= {p}  :", s2)
print("Counter lẻ:", odd_by_counter)
print("Tập góc lẻ của ex1:", sorted(toggle_set([p for r in ex1 for p in corners(r)])))

toggle    : {(1, 1), (3, 3)}
s ^= {p}  : {(1, 1), (3, 3)}
Counter lẻ: {(1, 1), (3, 3)}
Tập góc lẻ của ex1: [(1, 1), (1, 4), (4, 1), (4, 4)]


### 9.3 Chỉ kiểm điều kiện góc có đủ không? — KHÔNG (phản ví dụ 1)

Xếp **3 hình giống hệt** `[0,0,1,1]` chồng lên nhau. Mỗi góc gặp 3 lần → lẻ → còn trong set. Tập góc lẻ = `{(0,0), (1,0), (0,1), (1,1)}` = đúng 4 góc của hình bao `[0,0,1,1]`! Điều kiện góc **qua**, nhưng rõ ràng đây là chồng 3 lớp.

Điều kiện diện tích bắt được nó: `Σ area = 3 ≠ 1 = area(box)`.

Bài học: số lần chồng **lẻ** (3, 5, …) "lừa" được phép đếm chẵn lẻ. Ta **cần** thêm diện tích.

In [15]:
cx_corner = [[0, 0, 1, 1], [0, 0, 1, 1], [0, 0, 1, 1]]
draw(cx_corner, "cx_corner: 3 hình trùng nhau")
odd = toggle_set([p for r in cx_corner for p in corners(r)])
box = bounding_box(cx_corner)
print("tập góc lẻ   :", sorted(odd))
print("4 góc bao    :", sorted(corners(box)))
print("tổng diện tích =", sum(area(r) for r in cx_corner), "| diện tích bao =", area(box))
print("đáp án đúng (brute force):", is_cover_cells(cx_corner))

cx_corner: 3 hình trùng nhau
  y= 0 | 3
         +--
      x =   0
tập góc lẻ   : [(0, 0), (0, 1), (1, 0), (1, 1)]
4 góc bao    : [(0, 0), (0, 1), (1, 0), (1, 1)]
tổng diện tích = 3 | diện tích bao = 1
đáp án đúng (brute force): False


### 9.4 Chỉ kiểm diện tích có đủ không? — KHÔNG (phản ví dụ 2)

Đây là cái bẫy quan trọng nhất: **một chỗ chồng có thể "bù" đúng một chỗ hở**.

Xét `cx_area = [[0,0,2,1], [0,0,1,1], [1,1,2,2]]` trong hình bao `[0,0,2,2]`:

- `A = [0,0,2,1]` (diện tích 2) và `B = [0,0,1,1]` (diện tích 1) **chồng** ở ô `(0,0)`.
- `C = [1,1,2,2]` (diện tích 1). Ô `(0,1)` **không ai phủ** → hở.
- `Σ area = 2 + 1 + 1 = 4 = area(box)` → diện tích khớp hoàn hảo, nhưng đáp án là `False`!

Điều kiện góc bắt được nó: tập góc lẻ là `{(1,0), (2,0), (1,2), (2,2)}`, không phải `{(0,0), (2,0), (0,2), (2,2)}`.

In [16]:
cx_area = [[0, 0, 2, 1], [0, 0, 1, 1], [1, 1, 2, 2]]
draw(cx_area, "cx_area: chồng 1 ô + hở 1 ô")
odd = toggle_set([p for r in cx_area for p in corners(r)])
box = bounding_box(cx_area)
print("tổng diện tích =", sum(area(r) for r in cx_area), "| diện tích bao =", area(box))
print("tập góc lẻ   :", sorted(odd))
print("4 góc bao    :", sorted(corners(box)))
print("đáp án đúng (brute force):", is_cover_cells(cx_area))

cx_area: chồng 1 ô + hở 1 ô
  y= 1 | . C
  y= 0 | 2 A
         +----
      x =   0 1
tổng diện tích = 4 | diện tích bao = 4
tập góc lẻ   : [(1, 0), (1, 2), (2, 0), (2, 2)]
4 góc bao    : [(0, 0), (0, 2), (2, 0), (2, 2)]
đáp án đúng (brute force): False


### 9.5 Hai điều kiện CÙNG NHAU thì đủ — Bổ đề "đếm góc dưới-trái"

Ta đã thấy mỗi điều kiện riêng lẻ đều thủng. Giờ chứng minh: **diện tích khớp + tập góc lẻ đúng bằng 4 góc bao ⇒ phủ hoàn hảo**. Chìa khóa là một bổ đề rất dễ kiểm bằng tay:

> **Bổ đề.** Lấy một hình chữ nhật `R` và một điểm `p` **không nằm trên đường lưới nào** (ví dụ tâm ô `(i + 0.5, j + 0.5)`). Đếm số góc của `R` nằm **dưới-trái** `p` (tức là góc `(cx, cy)` có `cx < p.x` và `cy < p.y`). Khi đó: **`p` nằm trong `R` ⇔ con số đó lẻ** (chính xác là bằng 1).

Kiểm tay với `R = [1,1,3,3]`:

| Vị trí của `p` | Các góc dưới-trái | Số góc | Lẻ? | `p` trong `R`? |
|---|---|:---:|:---:|:---:|
| `(2.5, 2.5)` bên trong | `(1,1)` | 1 | ✅ | ✅ |
| `(3.5, 1.5)` bên phải | `(1,1), (3,1)` | 2 | ❌ | ❌ |
| `(1.5, 3.5)` phía trên | `(1,1), (1,3)` | 2 | ❌ | ❌ |
| `(3.5, 3.5)` trên-phải | cả 4 góc | 4 | ❌ | ❌ |
| `(0.5, 2.5)` bên trái | không có | 0 | ❌ | ❌ |

Lý do tổng quát: một góc `(cx, cy)` nằm dưới-trái `p` khi `cx < p.x` **và** `cy < p.y`. Hoành độ góc chỉ có thể là `x` hoặc `a`, tung độ chỉ có thể là `y` hoặc `b`, nên

```text
số góc dưới-trái p = (số giá trị trong {x, a} nhỏ hơn p.x) × (số giá trị trong {y, b} nhỏ hơn p.y)
```

- Thừa số thứ nhất bằng `0` nếu `p` ở bên trái `R`, `1` nếu `p` nằm giữa `x` và `a`, `2` nếu `p` ở bên phải.
- Thừa số thứ hai tương tự theo chiều dọc.

Tích hai số là **lẻ** khi và chỉ khi **cả hai** đều lẻ, tức là cả hai đều bằng `1`: `p` nằm giữa `x..a` **và** giữa `y..b` — chính là `p` nằm trong `R`. Bảng trên: `1×1 = 1`, `2×1 = 2`, `1×2 = 2`, `2×2 = 4`, `0×1 = 0`.

Cell dưới in "số góc dưới-trái" cho mọi tâm ô trong lưới `5 × 5`: các số `1` tạo đúng khối `2 × 2` của `R`.

In [17]:
def corners_lower_left(rect, px, py):
    return sum(1 for cx, cy in corners(rect) if cx < px and cy < py)

R = [1, 1, 3, 3]
print("Số góc của R = [1,1,3,3] nằm dưới-trái tâm mỗi ô:")
for j in range(4, -1, -1):
    row = [corners_lower_left(R, i + 0.5, j + 0.5) for i in range(5)]
    print(f"  y={j} |", " ".join(map(str, row)))
print("        x = 0 1 2 3 4")

Số góc của R = [1,1,3,3] nằm dưới-trái tâm mỗi ô:
  y=4 | 0 2 2 4 4
  y=3 | 0 2 2 4 4
  y=2 | 0 1 1 2 2
  y=1 | 0 1 1 2 2
  y=0 | 0 0 0 0 0
        x = 0 1 2 3 4


### 9.6 Ghép bổ đề lại thành chứng minh

Gọi `f(p)` = **số hình phủ điểm `p`** (với `p` là tâm một ô). Theo bổ đề, với **từng** hình `R`: "`R` phủ `p`" (đếm là 1, không phủ đếm là 0) có cùng chẵn lẻ với "số góc của `R` nằm dưới-trái `p`". Cộng hai vế trên **mọi** hình (tổng các số cùng chẵn lẻ thì cũng cùng chẵn lẻ):

```text
f(p) ≡ Σ_R (số góc của R dưới-trái p)       (mod 2)
     ≡ Σ_{điểm q dưới-trái p} (số lần q là góc)   (mod 2)
     ≡ (số điểm góc LẺ nằm dưới-trái p)       (mod 2)
```

(dòng 3: các điểm xuất hiện chẵn lần đóng góp số chẵn, không đổi chẵn lẻ — tức là chỉ những điểm còn lại trong **set bật/tắt** mới quan trọng.)

Giờ giả sử set bật/tắt chỉ còn đúng 4 góc bao `(X,Y), (A,Y), (X,B), (A,B)`:

- **Không hở:** với `p` bên trong hình bao, trong 4 góc bao chỉ có `(X, Y)` nằm dưới-trái `p` → `f(p)` **lẻ** → `f(p) >= 1`. Mọi ô trong hình bao đều được phủ!
- **Không chồng:** `Σ area(r)` chính là tổng `f(p)` trên mọi ô của hình bao (mỗi ô đóng góp đúng số lớp phủ nó). Mỗi ô có `f(p) >= 1`, mà tổng lại bằng `area(box)` = số ô → **mọi** ô có `f(p) = 1` đúng → không chồng.

Ngược lại (điều kiện **cần**) đã chứng minh ở 9.1. Vậy:

> **Phủ hoàn hảo ⇔ (`Σ area == area(box)`) VÀ (tập góc lẻ == 4 góc của hình bao).**

Đối chiếu với các phản ví dụ: ở `cx_corner` mọi ô có `f = 3` (lẻ, qua điều kiện góc) nhưng diện tích lộ ra; ở `cx_area` có ô `f = 2` (chẵn!) và ô `f = 0` — chính hai ô "bất thường" này làm tập góc lẻ bị lệch. Cell dưới in `f(p)` thật (brute force) và chẵn lẻ **dự đoán** từ tập góc lẻ cho ex3 và `cx_area` — hai lưới luôn khớp nhau theo mod 2.

In [18]:
def coverage_grid(rects):
    X, Y, A, B = bounding_box(rects)
    return [[sum(1 for (x, y, a, b) in rects if x <= i < a and y <= j < b)
             for i in range(X, A)] for j in range(B - 1, Y - 1, -1)]

def parity_prediction(rects):
    X, Y, A, B = bounding_box(rects)
    odd = toggle_set([p for r in rects for p in corners(r)])
    return [[sum(1 for (cx, cy) in odd if cx < i + 0.5 and cy < j + 0.5) % 2
             for i in range(X, A)] for j in range(B - 1, Y - 1, -1)]

for name, rs in [("ex3", ex3), ("cx_area", cx_area)]:
    print(f"{name}:  f(p) thật   |  chẵn lẻ dự đoán từ góc lẻ")
    for row_f, row_p in zip(coverage_grid(rs), parity_prediction(rs)):
        print("        ", " ".join(map(str, row_f)), "   |  ", " ".join(map(str, row_p)))

ex3:  f(p) thật   |  chẵn lẻ dự đoán từ góc lẻ
         1 1 1    |   1 1 1
         1 2 1    |   1 0 1
         1 1 1    |   1 1 1
cx_area:  f(p) thật   |  chẵn lẻ dự đoán từ góc lẻ
         0 1    |   0 1
         2 1    |   0 1


## 10. Cách 3 (lời giải chính) — Diện tích + tập góc bật/tắt, `O(n)`

Ghép tất cả lại, thuật toán chỉ cần **một lần duyệt** qua các hình:

1. Khởi tạo `X, Y = +∞`, `A, B = −∞` (sẽ thành hình bao), `total = 0`, `corner_set = set()`.
2. Với mỗi hình `[x, y, a, b]`:
   - cập nhật hình bao: `X = min(X, x)`, `Y = min(Y, y)`, `A = max(A, a)`, `B = max(B, b)`;
   - cộng diện tích: `total += (a − x) · (b − y)`;
   - **bật/tắt** 4 góc `(x,y), (a,y), (x,b), (a,b)` trong `corner_set`.
3. Trả `True` khi và chỉ khi:
   - `total == (A − X) · (B − Y)`, **và**
   - `corner_set == {(X,Y), (A,Y), (X,B), (A,B)}` (so sánh **cả tập**, tức là vừa có đủ 4 góc bao, vừa **không có điểm thừa**).

Thứ tự kiểm hai điều kiện không quan trọng; kiểm diện tích trước thì rẻ hơn một chút.

In [19]:
def is_rectangle_cover_corners(rects):
    X = Y = float("inf")
    A = B = float("-inf")
    total = 0
    corner_set = set()
    for x, y, a, b in rects:
        X, Y = min(X, x), min(Y, y)
        A, B = max(A, a), max(B, b)
        total += (a - x) * (b - y)
        for p in ((x, y), (a, y), (x, b), (a, b)):
            if p in corner_set:
                corner_set.remove(p)        # gặp số chẵn lần -> tắt
            else:
                corner_set.add(p)           # gặp số lẻ lần -> bật
    if total != (A - X) * (B - Y):
        return False                        # diện tích lệch -> chắc chắn hở hoặc chồng
    return corner_set == {(X, Y), (A, Y), (X, B), (A, B)}

In [20]:
cases = [("ex1", ex1, True), ("ex2", ex2, False), ("ex3", ex3, False),
         ("cx_corner", cx_corner, False), ("cx_area", cx_area, False)]
for name, rs, expected in cases:
    got = is_rectangle_cover_corners(rs)
    print(f"{name:10s} -> {got}  (mong đợi {expected})")
    assert got == expected

ex1        -> True  (mong đợi True)
ex2        -> False  (mong đợi False)
ex3        -> False  (mong đợi False)
cx_corner  -> False  (mong đợi False)
cx_area    -> False  (mong đợi False)


## 11. Trace từng bước

### 11.1 Ex1 (kết quả `True`)

Hàm `trace_corners` in, cho mỗi hình: tên, diện tích, hành động với từng góc (`+` = bật/thêm, `−` = tắt/xóa), kích thước set sau đó và tổng diện tích tích lũy. Kết quả mong đợi (tự tính tay trước khi chạy):

| Hình | Diện tích | 4 góc | Hành động | `|set|` sau | Tổng S |
|---|:---:|---|---|:---:|:---:|
| `A=[1,1,3,3]` | 4 | `(1,1) (3,1) (1,3) (3,3)` | `+ + + +` | 4 | 4 |
| `B=[3,1,4,2]` | 1 | `(3,1) (4,1) (3,2) (4,2)` | `− + + +` | 6 | 5 |
| `C=[3,2,4,4]` | 2 | `(3,2) (4,2) (3,4) (4,4)` | `− − + +` | 6 | 7 |
| `D=[1,3,2,4]` | 1 | `(1,3) (2,3) (1,4) (2,4)` | `− + + +` | 8 | 8 |
| `E=[2,3,3,4]` | 1 | `(2,3) (3,3) (2,4) (3,4)` | `− − − −` | 4 | 9 |

Hình `E` là "mảnh ghép cuối": cả 4 góc của nó đều đã được bật bởi hàng xóm, nên nó tắt hết. Còn lại `{(1,1), (4,1), (1,4), (4,4)}` = đúng 4 góc của hình bao `[1,1,4,4]`, và `9 == 3·3` → `True`.

Để ý: kích thước set **lên xuống** trong quá trình — chỉ trạng thái **cuối cùng** mới có ý nghĩa.

In [21]:
def fmt(p):
    return f"({p[0]},{p[1]})"

def trace_corners(rects):
    corner_set = set()
    total = 0
    for k, (x, y, a, b) in enumerate(rects):
        s = (a - x) * (b - y)
        total += s
        acts = []
        for p in ((x, y), (a, y), (x, b), (a, b)):
            if p in corner_set:
                corner_set.remove(p); acts.append("-" + fmt(p))
            else:
                corner_set.add(p); acts.append("+" + fmt(p))
        print(f"{chr(65 + k)}={[x, y, a, b]} S={s} | {' '.join(acts)} | |set|={len(corner_set)} | tổng S={total}")
    X, Y, A, B = bounding_box(rects)
    box_corners = {(X, Y), (A, Y), (X, B), (A, B)}
    print("set cuối    :", " ".join(fmt(p) for p in sorted(corner_set)))
    print("4 góc bao   :", " ".join(fmt(p) for p in sorted(box_corners)))
    print(f"diện tích: {total} vs {(A - X) * (B - Y)} -> {total == (A - X) * (B - Y)} | "
          f"góc khớp -> {corner_set == box_corners}")
    return total == (A - X) * (B - Y) and corner_set == box_corners

print("Kết quả:", trace_corners(ex1))

A=[1, 1, 3, 3] S=4 | +(1,1) +(3,1) +(1,3) +(3,3) | |set|=4 | tổng S=4
B=[3, 1, 4, 2] S=1 | -(3,1) +(4,1) +(3,2) +(4,2) | |set|=6 | tổng S=5
C=[3, 2, 4, 4] S=2 | -(3,2) -(4,2) +(3,4) +(4,4) | |set|=6 | tổng S=7
D=[1, 3, 2, 4] S=1 | -(1,3) +(2,3) +(1,4) +(2,4) | |set|=8 | tổng S=8
E=[2, 3, 3, 4] S=1 | -(2,3) -(3,3) -(2,4) -(3,4) | |set|=4 | tổng S=9
set cuối    : (1,1) (1,4) (4,1) (4,4)
4 góc bao   : (1,1) (1,4) (4,1) (4,4)
diện tích: 9 vs 9 -> True | góc khớp -> True
Kết quả: True


### 11.2 Ex3 (chồng) và `cx_area` (chồng bù hở)

Với ex3, ba hình đầu giống ex1 (`A, B` giống hệt; `C=[1,3,2,4]` chính là `D` của ex1). Hình cuối `D=[2,2,4,4]` khác hẳn:

| Hình | Diện tích | Hành động | `|set|` sau | Tổng S |
|---|:---:|---|:---:|:---:|
| `A=[1,1,3,3]` | 4 | `+(1,1) +(3,1) +(1,3) +(3,3)` | 4 | 4 |
| `B=[3,1,4,2]` | 1 | `−(3,1) +(4,1) +(3,2) +(4,2)` | 6 | 5 |
| `C=[1,3,2,4]` | 1 | `−(1,3) +(2,3) +(1,4) +(2,4)` | 8 | 6 |
| `D=[2,2,4,4]` | 4 | `+(2,2) −(4,2) −(2,4) +(4,4)` | 8 | 10 |

Hỏng **cả hai** điều kiện: `10 ≠ 9` và set còn 8 điểm (thừa `(2,2), (2,3), (3,2), (3,3)` — chính là 4 góc của **ô bị chồng** `[2,2,3,3]`; nhìn lại hình vẽ ex3, ô `(2,2)` mang số `2`). Ex3 không có chỗ hở, chỉ thừa đúng 1 đơn vị diện tích do chồng.

Còn với `cx_area`, diện tích khớp (`4 == 4`) nên chỉ **điều kiện góc** cứu ta.

In [22]:
print("Kết quả ex3:", trace_corners(ex3))
print()
print("Kết quả cx_area:", trace_corners(cx_area))

A=[1, 1, 3, 3] S=4 | +(1,1) +(3,1) +(1,3) +(3,3) | |set|=4 | tổng S=4
B=[3, 1, 4, 2] S=1 | -(3,1) +(4,1) +(3,2) +(4,2) | |set|=6 | tổng S=5
C=[1, 3, 2, 4] S=1 | -(1,3) +(2,3) +(1,4) +(2,4) | |set|=8 | tổng S=6
D=[2, 2, 4, 4] S=4 | +(2,2) -(4,2) -(2,4) +(4,4) | |set|=8 | tổng S=10
set cuối    : (1,1) (1,4) (2,2) (2,3) (3,2) (3,3) (4,1) (4,4)
4 góc bao   : (1,1) (1,4) (4,1) (4,4)
diện tích: 10 vs 9 -> False | góc khớp -> False
Kết quả ex3: False

A=[0, 0, 2, 1] S=2 | +(0,0) +(2,0) +(0,1) +(2,1) | |set|=4 | tổng S=2
B=[0, 0, 1, 1] S=1 | -(0,0) +(1,0) -(0,1) +(1,1) | |set|=4 | tổng S=3
C=[1, 1, 2, 2] S=1 | -(1,1) -(2,1) +(1,2) +(2,2) | |set|=4 | tổng S=4
set cuối    : (1,0) (1,2) (2,0) (2,2)
4 góc bao   : (0,0) (0,2) (2,0) (2,2)
diện tích: 4 vs 4 -> True | góc khớp -> False
Kết quả cx_area: False


## 12. Code nộp LeetCode

Chuyển hàm `is_rectangle_cover_corners` vào `class Solution`, đúng signature. Vài chi tiết:

- LeetCode Python3 đã `from typing import List` sẵn; trong notebook ta tự import (đã làm ở cell đầu).
- Thay vì `float("inf")`, có thể khởi tạo hình bao bằng chính hình đầu tiên — `n >= 1` nên luôn có hình đầu tiên. Tránh trộn `float` với `int` cho gọn.
- Dùng `corner_set ^= {p}` hoặc `if/else` đều được; `if/else` dễ đọc hơn cho người mới.
- Python số nguyên không tràn. Nếu viết C++/Java: diện tích tới `4·10^10` **vượt** `int` 32-bit → phải dùng `long long` / `long`.

In [23]:
class Solution:
    def isRectangleCover(self, rectangles: List[List[int]]) -> bool:
        X, Y, A, B = rectangles[0]
        total = 0
        corner_set = set()
        for x, y, a, b in rectangles:
            X, Y = min(X, x), min(Y, y)
            A, B = max(A, a), max(B, b)
            total += (a - x) * (b - y)
            for p in ((x, y), (a, y), (x, b), (a, b)):
                if p in corner_set:
                    corner_set.remove(p)
                else:
                    corner_set.add(p)
        if total != (A - X) * (B - Y):
            return False
        return corner_set == {(X, Y), (A, Y), (X, B), (A, B)}

In [24]:
sol = Solution()
pinwheel = [[0, 0, 2, 1], [2, 0, 3, 2], [1, 2, 3, 3], [0, 1, 1, 3], [1, 1, 2, 2]]
cx_bbox = [[0, 0, 3, 1], [0, 1, 2, 3], [1, 2, 3, 3]]     # dùng ở mục lỗi thường gặp

tests = [
    (ex1, True), (ex2, False), (ex3, False),                       # ví dụ của đề
    ([[0, 0, 1, 1]], True),                                        # n = 1
    ([[-100000, -100000, 100000, 100000]], True),                  # tọa độ biên
    ([[0, 0, 1, 1], [0, 0, 1, 1]], False),                         # 2 hình trùng nhau
    (cx_corner, False),                                            # chồng 3 lớp
    (cx_area, False),                                              # chồng bù hở
    (cx_bbox, False),                                              # đủ 4 góc bao + điểm thừa
    ([[0, 0, 1, 1], [1, 1, 2, 2]], False),                         # chỉ chạm góc -> hở
    ([[0, 0, 2, 1], [0, 1, 1, 2]], False),                         # hình chữ L
    ([[0, 0, 1, 1], [2, 0, 3, 1]], False),                         # hai mảnh rời
    (pinwheel, True),                                              # "chong chóng": không cắt thẳng được
    (grid_cover(3, 4, -5, -7), True),                              # lưới ô vuông, tọa độ âm
]
for rs, expected in tests:
    assert sol.isRectangleCover(rs) == expected, (rs, expected)
print(f"Tất cả {len(tests)} test PASS")
draw(pinwheel, "pinwheel (true):")

Tất cả 14 test PASS
pinwheel (true):
  y= 2 | D C C
  y= 1 | D E B
  y= 0 | A A B
         +------
      x =   0 1 2


## 13. Cách 4 (nâng cao, tùy chọn) — Sweep line

Cách 3 đã đủ để nộp bài. Nhưng có một kỹ thuật tổng quát hơn rất đáng học, vì nó giải được cả những bài mà "mẹo góc" không áp dụng được (ví dụ tính diện tích hợp, bài Skyline): **sweep line** (đường quét).

Ý tưởng: tưởng tượng một **đường thẳng đứng** quét từ trái sang phải. Ta chỉ dừng lại ở những **sự kiện** (event) — nơi có hình bắt đầu hoặc kết thúc — và duy trì **trạng thái** của những gì đường quét đang cắt qua.

Ví dụ mini 1 chiều: các cuộc họp `[1,4], [2,5], [4,6], [7,8]`. Hỏi lúc đông nhất có bao nhiêu cuộc họp cùng diễn ra?

- Mỗi cuộc họp tạo 2 sự kiện: `(bắt đầu, +1)` và `(kết thúc, −1)`.
- Sắp xếp sự kiện theo thời gian; **cùng thời điểm thì kết thúc trước bắt đầu** (cuộc họp `[1,4]` kết thúc đúng lúc `[4,6]` bắt đầu → không trùng). Với tuple `(t, delta)`, `−1 < +1` nên `sort()` tự xếp kết thúc trước.
- Quét và cộng dồn `delta`, ghi lại giá trị lớn nhất.

In [25]:
meetings = [[1, 4], [2, 5], [4, 6], [7, 8]]
events = []
for s, e in meetings:
    events.append((s, +1))
    events.append((e, -1))
events.sort()

active = best = 0
for t, delta in events:
    active += delta
    best = max(best, active)
    print(f"t={t}: {'bắt đầu' if delta > 0 else 'kết thúc'} -> đang họp {active}")
print("Đông nhất:", best)

t=1: bắt đầu -> đang họp 1
t=2: bắt đầu -> đang họp 2
t=4: kết thúc -> đang họp 1
t=4: bắt đầu -> đang họp 2
t=5: kết thúc -> đang họp 1
t=6: kết thúc -> đang họp 0
t=7: bắt đầu -> đang họp 1
t=8: kết thúc -> đang họp 0
Đông nhất: 2


### 13.1 Kiến thức nền 7 — Module `bisect`: tìm vị trí chèn trong list đã sắp xếp

Khi sweep cho bài này, trạng thái là **danh sách các đoạn `y` đang bị đường quét cắt**, giữ **luôn sắp xếp**. Python có module `bisect` (tìm kiếm nhị phân — binary search) giúp việc này:

- `bisect.bisect_left(L, v)`: vị trí **đầu tiên** để chèn `v` mà `L` vẫn sắp xếp — mọi phần tử bên trái `< v`, bên phải `>= v`. Mất `O(log n)`.
- Sau khi có vị trí `i`, **hàng xóm** của `v` là `L[i − 1]` (lớn nhất mà nhỏ hơn) và `L[i]` (nhỏ nhất mà không nhỏ hơn).
- `L.insert(i, v)` chèn, `L.pop(i)` xóa (cả hai `O(len(L))` vì phải dời phần tử, nhưng rất nhanh trong thực tế).
- Tuple so sánh theo **từ điển**: `(1, 3) < (2, 4)` vì `1 < 2`; `(1, 2) < (1, 3)` vì phần đầu bằng nhau, so tiếp `2 < 3`.

In [26]:
L = [(1, 3), (5, 6), (8, 9)]
v = (4, 5)
i = bisect.bisect_left(L, v)
print("chèn", v, "vào vị trí", i, "| hàng xóm trái:", L[i - 1] if i > 0 else None,
      "| hàng xóm phải:", L[i] if i < len(L) else None)
L.insert(i, v)
print("sau khi chèn:", L)
print("bisect_left(L, (1, 3)) =", bisect.bisect_left(L, (1, 3)), "-> tìm đúng chỗ của phần tử đã có")

chèn (4, 5) vào vị trí 1 | hàng xóm trái: (1, 3) | hàng xóm phải: (5, 6)
sau khi chèn: [(1, 3), (4, 5), (5, 6), (8, 9)]
bisect_left(L, (1, 3)) = 0 -> tìm đúng chỗ của phần tử đã có


### 13.2 Sweep line cho Perfect Rectangle

Quét theo trục `x`:

- Mỗi hình `[x, y, a, b]` sinh 2 sự kiện: tại `x` **thêm** đoạn `(y, b)`, tại `a` **bỏ** đoạn `(y, b)`. Mã hóa `(hoành độ, loại, y, b)` với loại `0` = bỏ, `1` = thêm → `sort()` tự xếp **bỏ trước thêm** khi cùng hoành độ (hai hình ghép sát theo cạnh dọc không bị coi là chồng).
- `active` = list các đoạn `y` đang mở, **luôn sắp xếp và đôi một rời nhau** (bất biến — invariant).
- Khi **thêm** đoạn `(y1, y2)`: tìm `i = bisect_left(active, (y1, y2))`. Vì các đoạn trong `active` rời nhau và sắp xếp, đoạn mới chồng với **ai đó** thì chắc chắn chồng với **hàng xóm** của nó:
  - hàng xóm trái `active[i−1]` chồng nếu `active[i−1][1] > y1` (kết thúc sau khi đoạn mới bắt đầu);
  - hàng xóm phải `active[i]` chồng nếu `active[i][0] < y2` (bắt đầu trước khi đoạn mới kết thúc).
  - Có chồng → hai hình chồng nhau (cùng cắt đường quét ở một khoảng `y` dương, và cả hai đều kéo dài sang phải một chút) → `False`.
- Khi **bỏ**: tìm đúng vị trí bằng `bisect_left` rồi `pop`.
- Quét xong không chồng → còn kiểm **diện tích** như cách 2 để loại trường hợp hở.

Đây thực chất là **cách 2 được tăng tốc**: thay vì so mọi cặp, ta chỉ so mỗi hình mới với **2 hàng xóm** tại thời điểm nó xuất hiện.

In [27]:
def is_cover_sweep(rects):
    events = []
    for x, y, a, b in rects:
        events.append((x, 1, y, b))      # thêm đoạn [y, b] tại x
        events.append((a, 0, y, b))      # bỏ đoạn [y, b] tại a (0 < 1 -> bỏ trước)
    events.sort()
    active = []                          # các đoạn y đang mở, sắp xếp, rời nhau
    for _, kind, y1, y2 in events:
        i = bisect.bisect_left(active, (y1, y2))
        if kind == 0:
            active.pop(i)
        else:
            if i > 0 and active[i - 1][1] > y1:
                return False             # chồng với hàng xóm dưới
            if i < len(active) and active[i][0] < y2:
                return False             # chồng với hàng xóm trên
            active.insert(i, (y1, y2))
    total = sum(area(r) for r in rects)
    return total == area(bounding_box(rects))

for name, rs in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3), ("cx_corner", cx_corner), ("cx_area", cx_area)]:
    print(f"{name:10s} -> {is_cover_sweep(rs)}")

ex1        -> True
ex2        -> False
ex3        -> False
cx_corner  -> False
cx_area    -> False


### 13.3 Trace sweep line

Với ex1, ở `x = 2` hình `D=[1,3,2,4]` kết thúc **đúng lúc** `E=[2,3,3,4]` bắt đầu với cùng đoạn `(3,4)` — nhờ "bỏ trước thêm", không bị báo chồng. Ở `x = 3`, `A` và `E` cùng kết thúc rồi `B, C` mới vào.

Với ex3, ở `x = 2`: `C=[1,3,2,4]` bị bỏ, `active = [(1,3)]` (đoạn của `A`); `D=[2,2,4,4]` thêm đoạn `(2,4)` → `i = 1`, hàng xóm dưới `(1,3)` kết thúc ở `3 > 2` → **chồng**.

In [28]:
def trace_sweep(rects):
    events = []
    for k, (x, y, a, b) in enumerate(rects):
        events.append((x, 1, y, b, chr(65 + k)))
        events.append((a, 0, y, b, chr(65 + k)))
    events.sort()
    active = []
    for xx, kind, y1, y2, name in events:
        i = bisect.bisect_left(active, (y1, y2))
        if kind == 0:
            active.pop(i)
            print(f"x={xx}: bỏ  {name} ({y1},{y2}) -> active={active}")
        else:
            if (i > 0 and active[i - 1][1] > y1) or (i < len(active) and active[i][0] < y2):
                print(f"x={xx}: thêm {name} ({y1},{y2}) CHỒNG với active={active} -> False")
                return False
            active.insert(i, (y1, y2))
            print(f"x={xx}: thêm {name} ({y1},{y2}) -> active={active}")
    print("Không chồng; còn kiểm diện tích:", sum(area(r) for r in rects), "vs", area(bounding_box(rects)))
    return sum(area(r) for r in rects) == area(bounding_box(rects))

print("ex1:", trace_sweep(ex1))
print()
print("ex3:", trace_sweep(ex3))

x=1: thêm A (1,3) -> active=[(1, 3)]
x=1: thêm D (3,4) -> active=[(1, 3), (3, 4)]
x=2: bỏ  D (3,4) -> active=[(1, 3)]
x=2: thêm E (3,4) -> active=[(1, 3), (3, 4)]
x=3: bỏ  A (1,3) -> active=[(3, 4)]
x=3: bỏ  E (3,4) -> active=[]
x=3: thêm B (1,2) -> active=[(1, 2)]
x=3: thêm C (2,4) -> active=[(1, 2), (2, 4)]
x=4: bỏ  B (1,2) -> active=[(2, 4)]
x=4: bỏ  C (2,4) -> active=[]
Không chồng; còn kiểm diện tích: 9 vs 9
ex1: True

x=1: thêm A (1,3) -> active=[(1, 3)]
x=1: thêm C (3,4) -> active=[(1, 3), (3, 4)]
x=2: bỏ  C (3,4) -> active=[(1, 3)]
x=2: thêm D (2,4) CHỒNG với active=[(1, 3)] -> False
ex3: False


## 14. Test đối chiếu chéo trên input ngẫu nhiên

Trên tọa độ nhỏ, cách 1 (tô ô) là trọng tài tin cậy tuyệt đối. Nhưng nếu chỉ sinh các hình ngẫu nhiên thuần túy, gần như **không bao giờ** ra trường hợp `True`, nên test sẽ yếu. Vì vậy ta sinh dữ liệu theo hai bước:

1. **Sinh phủ hoàn hảo** bằng cách cắt đệ quy (guillotine cut): lấy một hình chữ nhật, ngẫu nhiên cắt dọc hoặc ngang tại một tọa độ nguyên, rồi cắt tiếp từng nửa. Kết quả luôn là phủ hoàn hảo.
2. **Phá ngẫu nhiên** (mutate): giữ nguyên, xóa một hình, nhân đôi một hình, dịch một cạnh đi `±1`, hoặc thêm một hình ngẫu nhiên — rồi xáo trộn thứ tự.

Sau đó so 5 hàm: `is_cover_cells`, `is_cover_pairwise`, `is_rectangle_cover_corners`, `is_cover_sweep`, `Solution().isRectangleCover`. Hạt giống ngẫu nhiên cố định (`random.seed`) để kết quả lặp lại được.

In [29]:
def random_perfect(x0, y0, x1, y1, rng, depth=0):
    w, h = x1 - x0, y1 - y0
    if depth >= 4 or (w == 1 and h == 1) or rng.random() < 0.2:
        return [[x0, y0, x1, y1]]
    if w > 1 and (h == 1 or rng.random() < 0.5):
        c = rng.randint(x0 + 1, x1 - 1)                       # cắt dọc
        return random_perfect(x0, y0, c, y1, rng, depth + 1) + random_perfect(c, y0, x1, y1, rng, depth + 1)
    c = rng.randint(y0 + 1, y1 - 1)                           # cắt ngang
    return random_perfect(x0, y0, x1, c, rng, depth + 1) + random_perfect(x0, c, x1, y1, rng, depth + 1)

def mutate(rects, rng):
    rs = [r[:] for r in rects]
    op = rng.choice(["giữ", "giữ", "xóa", "nhân đôi", "dịch", "thêm"])
    if op == "xóa" and len(rs) > 1:
        rs.pop(rng.randrange(len(rs)))
    elif op == "nhân đôi":
        rs.append(rng.choice(rs)[:])
    elif op == "dịch":
        k, c, d = rng.randrange(len(rs)), rng.randrange(4), rng.choice([-1, 1])
        rs[k][c] += d
        if rs[k][0] >= rs[k][2] or rs[k][1] >= rs[k][3]:
            rs[k][c] -= d                                     # không cho hình suy biến
    elif op == "thêm":
        x, y = rng.randint(0, 5), rng.randint(0, 5)
        rs.append([x, y, x + rng.randint(1, 3), y + rng.randint(1, 3)])
    rng.shuffle(rs)
    return rs

rng = random.Random(391)
funcs = [is_cover_cells, is_cover_pairwise, is_rectangle_cover_corners, is_cover_sweep,
         Solution().isRectangleCover]
stats = Counter()
for _ in range(3000):
    x0, y0 = rng.randint(-3, 2), rng.randint(-3, 2)
    rs = mutate(random_perfect(x0, y0, x0 + rng.randint(1, 6), y0 + rng.randint(1, 6), rng), rng)
    answers = [f(rs) for f in funcs]
    assert len(set(answers)) == 1, (rs, answers)
    stats[answers[0]] += 1
print("3000 bộ test: 5 cách cho cùng đáp án. Phân bố:", dict(stats))

3000 bộ test: 5 cách cho cùng đáp án. Phân bố: {False: 1485, True: 1515}


In [30]:
# Hiệu năng ở cỡ lớn nhất: n = 20000 ô vuông ghép khít, tọa độ sát biên -10^5
big = grid_cover(100, 200, -100000, -100000)
random.seed(0)
random.shuffle(big)
big_bad = big + [big[0][:]]                  # thêm một hình trùng -> False

for name, f in [("cách 3 (góc)", Solution().isRectangleCover), ("cách 4 (sweep)", is_cover_sweep)]:
    for label, rs, expected in [("đúng", big, True), ("nhân đôi 1 hình", big_bad, False)]:
        t0 = time.perf_counter()
        got = f(rs)
        dt = time.perf_counter() - t0
        assert got == expected
        print(f"{name:15s} | n={len(rs)} {label:16s} -> {got} | {dt * 1000:.0f} ms")

cách 3 (góc)    | n=20000 đúng             -> True | 34 ms


cách 3 (góc)    | n=20001 nhân đôi 1 hình  -> False | 32 ms
cách 4 (sweep)  | n=20000 đúng             -> True | 42 ms
cách 4 (sweep)  | n=20001 nhân đôi 1 hình  -> False | 22 ms


## 15. Vì sao đúng — tóm tắt bất biến của từng cách

- **Cách 1 (tô ô):** bất biến "mỗi ô trong `painted` đã được tô đúng 1 lần". Gặp ô đã tô → chồng. Cuối cùng `painted ⊆` các ô của hình bao, nên `len(painted) == area(box)` ⇔ không thiếu ô nào.
- **Cách 2 (mọi cặp):** không cặp nào chồng ⇒ diện tích hợp = `Σ area`. Hợp nằm trong hình bao ⇒ `Σ area == area(box)` ⇔ hợp phủ kín hình bao (không hở).
- **Cách 3 (góc + diện tích):**
  - *Cần*: trong phủ hoàn hảo, quanh mỗi điểm `p`, các góc phần tư nằm trong hình bao được phủ đúng 1 lần ⇒ số hình nhận `p` làm góc là `4 − 2e − 4i` (bên trong), `2 − 2e` (trên cạnh bao), `1` (góc bao) ⇒ tập góc lẻ = 4 góc bao. Diện tích hiển nhiên khớp.
  - *Đủ*: bổ đề "đếm góc dưới-trái" cho `f(p) ≡ (số góc lẻ dưới-trái p) (mod 2)`. Chỉ còn 4 góc bao ⇒ mọi ô trong bao có `f` lẻ ⇒ `f >= 1` (không hở). `Σ f = Σ area = area(box)` mà mỗi `f >= 1` ⇒ mọi `f = 1` (không chồng).
  - Thiếu một trong hai điều kiện đều có phản ví dụ: `cx_corner` (chồng 3 lớp, góc qua) và `cx_area` (chồng bù hở, diện tích qua).
- **Cách 4 (sweep):** bất biến "các đoạn trong `active` sắp xếp và đôi một rời nhau" ⇒ đoạn mới chồng với ai thì chồng với hàng xóm của nó. Mọi cặp hình chồng nhau đều có lúc cùng nằm trong `active` (tại hoành độ bắt đầu của hình vào sau, nhờ bỏ-trước-thêm thì hai hình chỉ chạm không bị tính) ⇒ phát hiện được. Không chồng + diện tích khớp ⇒ đúng như cách 2.

## 16. Độ phức tạp và so sánh các cách

Ký hiệu: `n` = số hình chữ nhật (`<= 2·10^4`), `S` = tổng diện tích `Σ area(r)` (có thể tới cỡ `10^14`), `k` = số đoạn đang mở trong sweep (`k <= n`).

| Cách | Ý tưởng | Time | Space | Dùng khi |
|---|---|---|---|---|
| 1. Tô ô | `set` các ô đơn vị | `O(S)` | `O(S)` | tọa độ nhỏ, làm trọng tài |
| 2. Mọi cặp | `rects_overlap` + diện tích | `O(n²)` | `O(1)` thêm | `n` vài nghìn |
| **3. Góc + diện tích** | set bật/tắt 4 góc | **`O(n)`** trung bình | `O(n)` | **cách chính để nộp** |
| 4. Sweep line | sự kiện + `bisect` | `O(n log n + n·k)`; `O(n log n)` nếu dùng cây cân bằng | `O(n)` | tổng quát hóa sang bài khác |

Giải thích:

- Cách 3: mỗi hình làm 4 thao tác set (`O(1)` trung bình nhờ hash) + vài phép `min/max` → `O(n)`. Set chứa tối đa `4n` điểm. Lưu ý "trung bình": thao tác hash table trong trường hợp xấu nhất lý thuyết có thể chậm hơn, nhưng với tuple số nguyên thì thực tế luôn nhanh.
- Cách 4: sắp xếp `2n` sự kiện `O(n log n)`; mỗi `bisect` `O(log k)`, nhưng `list.insert/pop` phải dời phần tử `O(k)`. Đo ở mục 14 cho thấy với `n = 2·10^4` vẫn chỉ vài chục ms vì dời phần tử là thao tác bộ nhớ rất nhanh. Muốn chặt `O(n log n)` thì thay list bằng cây cân bằng (balanced BST / `SortedList` của thư viện `sortedcontainers` — không có trong thư viện chuẩn).

## 17. Lỗi thường gặp

| # | Lỗi | Ví dụ sai | Vì sao sai | Sửa |
|---|---|---|---|---|
| 1 | Quên kiểm diện tích | chỉ `return corner_set == box_corners` | `cx_corner` (3 hình trùng) có tập góc lẻ đúng 4 góc bao | luôn kiểm `total == area(box)` |
| 2 | Chỉ kiểm "4 góc bao **có trong** set" | `all(p in corner_set for p in box_corners)` | set còn điểm thừa vẫn qua, ví dụ `cx_bbox` | so **cả tập** `==` (hoặc thêm `len(corner_set) == 4`) |
| 3 | Chỉ kiểm diện tích | `return total == area(box)` | `cx_area`: chồng 1 ô bù đúng 1 ô hở | thêm điều kiện góc |
| 4 | Dùng `list` làm phần tử set | `corner_set.add([x, y])` | `list` không hashable → `TypeError` | dùng tuple `(x, y)` |
| 5 | Coi chạm là chồng | `max(...) <= min(...)` | ex1: `A` và `B` chỉ chạm cạnh `x = 3` → trả nhầm `False` | dùng `<` nghiêm ngặt |
| 6 | Sweep: thêm trước bỏ | sắp xếp với loại "thêm" `< ` loại "bỏ" | ex1: `D` kết thúc và `E` bắt đầu cùng ở `x = 2`, cùng đoạn `(3,4)` → báo chồng giả | mã hóa bỏ = 0, thêm = 1 |
| 7 | Tràn số (C++/Java) | `int area = (a-x)*(b-y)` | diện tích tới `4·10^10` | dùng `long` / `long long` (Python không bị) |

Cell dưới chạy thử các phiên bản sai 1, 2, 3, 5, 6 để thấy tận mắt chúng trả kết quả sai.

In [31]:
def collect(rects):
    total, cs = 0, set()
    for r in rects:
        total += area(r)
        for p in corners(r):
            cs ^= {p}
    X, Y, A, B = bounding_box(rects)
    return total, cs, {(X, Y), (A, Y), (X, B), (A, B)}, (A - X) * (B - Y)

def wrong1_no_area(rects):
    total, cs, bc, box_area = collect(rects)
    return cs == bc

def wrong2_contains_only(rects):
    total, cs, bc, box_area = collect(rects)
    return total == box_area and all(p in cs for p in bc)

def wrong3_area_only(rects):
    total, cs, bc, box_area = collect(rects)
    return total == box_area

def wrong5_touch_is_overlap(rects):
    for i in range(len(rects)):
        for j in range(i + 1, len(rects)):
            r1, r2 = rects[i], rects[j]
            if max(r1[0], r2[0]) <= min(r1[2], r2[2]) and max(r1[1], r2[1]) <= min(r1[3], r2[3]):
                return False
    return sum(area(r) for r in rects) == area(bounding_box(rects))

def wrong6_add_before_remove(rects):
    events = sorted([(x, 0, y, b) for x, y, a, b in rects] + [(a, 1, y, b) for x, y, a, b in rects])
    active = []
    for _, kind, y1, y2 in events:                   # kind 0 = thêm (SAI: thêm trước bỏ)
        i = bisect.bisect_left(active, (y1, y2))
        if kind == 1:
            active.pop(i)
        else:
            if (i > 0 and active[i - 1][1] > y1) or (i < len(active) and active[i][0] < y2):
                return False
            active.insert(i, (y1, y2))
    return sum(area(r) for r in rects) == area(bounding_box(rects))

for f, rs, name in [(wrong1_no_area, cx_corner, "cx_corner"), (wrong2_contains_only, cx_bbox, "cx_bbox"),
                    (wrong3_area_only, cx_area, "cx_area"), (wrong5_touch_is_overlap, ex1, "ex1"),
                    (wrong6_add_before_remove, ex1, "ex1")]:
    print(f"{f.__name__:26s} trên {name:9s}: trả {f(rs)!s:5s} | đúng là {is_cover_cells(rs)}")
draw(cx_bbox, "cx_bbox: đủ 4 góc bao, diện tích khớp, nhưng chồng + hở:")

wrong1_no_area             trên cx_corner: trả True  | đúng là False
wrong2_contains_only       trên cx_bbox  : trả True  | đúng là False
wrong3_area_only           trên cx_area  : trả True  | đúng là False
wrong5_touch_is_overlap    trên ex1      : trả False | đúng là True
wrong6_add_before_remove   trên ex1      : trả False | đúng là True
cx_bbox: đủ 4 góc bao, diện tích khớp, nhưng chồng + hở:
  y= 2 | B 2 C
  y= 1 | B B .
  y= 0 | A A A
         +------
      x =   0 1 2


## 18. Mẹo nhận dạng pattern

- Đề hỏi **"phủ kín, không chồng"** (exact cover / tiling) với các hình **song song trục** → nghĩ ngay tới **bất biến đếm được**: diện tích (bảo toàn tổng) + một đại lượng **chẵn lẻ** ở các "điểm đặc biệt" (góc, đầu mút). Diện tích bắt "thiếu/thừa về lượng", chẵn lẻ bắt "sai về hình dạng".
- Tọa độ lớn (`10^5`, `10^9`) mà `n` nhỏ hơn nhiều → **không được** làm theo từng ô; hãy làm theo **từng hình** (`O(n)`) hoặc theo **sự kiện** (sweep line), hoặc **nén tọa độ** (coordinate compression) nếu cần lưới.
- "Bật/tắt" trong `set` = XOR = đếm chẵn lẻ. Gặp đề có cụm "xuất hiện chẵn lần / lẻ lần" → nghĩ tới XOR (như bài *Single Number*) hoặc set toggle.
- Bài hình chữ nhật mà cần xử lý **giao nhau / hợp / chồng** tổng quát → **sweep line** theo một trục + cấu trúc dữ liệu giữ các đoạn đang mở ở trục kia (list + `bisect`, heap, segment tree).
- Luôn giữ một **brute force trên tọa độ nhỏ** để test: với bài hình học, phản ví dụ "bẫy" rất khó nghĩ ra bằng tay, nhưng random test tìm ra trong vài giây.

## 19. Biến thể và bài liên quan

- **Phiên bản 1 chiều**: các đoạn `[l, r]` có phủ chính xác `[min l, max r]` không? Cùng mẹo: tổng độ dài + tập đầu mút bật/tắt chỉ còn `{min l, max r}` (bài tập 4).
- **Đếm chính xác thay vì chẵn lẻ**: dùng `Counter`, yêu cầu 4 góc bao xuất hiện **đúng 1 lần** và mọi điểm khác xuất hiện **2 hoặc 4 lần** (đúng như phân tích ở 9.1). Kết hợp với diện tích, điều kiện này tương đương cách 3 — cell dưới kiểm chứng.
- **Tô ô + nén tọa độ**: chỉ giữ các hoành độ/tung độ thật sự xuất hiện, lưới còn tối đa `2n × 2n` ô "co giãn" → brute force `O(n²)`–`O(n³)`, đúng với mọi tọa độ nhưng vẫn chậm cho `n = 2·10^4`.
- Bài liên quan trên LeetCode:
  - **836. Rectangle Overlap** (Easy) — chính là hàm `rects_overlap`.
  - **223. Rectangle Area** (Medium) — diện tích hợp của 2 hình = tổng − phần chung.
  - **850. Rectangle Area II** (Hard) — diện tích hợp của nhiều hình: sweep line + nén tọa độ.
  - **218. The Skyline Problem** (Hard) — sweep line với các sự kiện bắt đầu/kết thúc.
  - **56. Merge Intervals** (Medium) — xử lý đoạn 1 chiều đã sắp xếp, nền tảng của sweep.

In [32]:
def is_cover_counter(rects):
    cnt = Counter()
    total = 0
    for r in rects:
        total += area(r)
        for p in corners(r):
            cnt[p] += 1
    X, Y, A, B = bounding_box(rects)
    if total != (A - X) * (B - Y):
        return False
    box_corners = {(X, Y), (A, Y), (X, B), (A, B)}
    for p, c in cnt.items():
        if p in box_corners:
            if c != 1:
                return False
        elif c not in (2, 4):
            return False
    return all(p in cnt for p in box_corners)

rng = random.Random(7)
for _ in range(2000):
    x0, y0 = rng.randint(-3, 2), rng.randint(-3, 2)
    rs = mutate(random_perfect(x0, y0, x0 + rng.randint(1, 6), y0 + rng.randint(1, 6), rng), rng)
    assert is_cover_counter(rs) == is_cover_cells(rs), rs
print("Phiên bản Counter (1 / 2 / 4 lần) khớp brute force trên 2000 bộ test")

Phiên bản Counter (1 / 2 / 4 lần) khớp brute force trên 2000 bộ test


## 20. Bài tập tự luyện

Làm theo thứ tự, từ dễ tới khó. **Đừng xem đáp án** ở cuối notebook trước khi tự thử.

**Bài 1 — Đoán output (tính tay).** Với `rects = [[0,0,1,1], [1,0,2,1], [0,1,2,2]]`, hãy vẽ hình ra giấy, rồi viết **set góc sau từng hình** khi chạy thuật toán bật/tắt, tổng diện tích, hình bao, và kết luận `True/False`.

**Bài 2 — Tính tay ví dụ 2 của đề.** Với ex2 `[[1,1,2,3],[1,3,2,4],[3,1,4,2],[3,2,4,4]]`: tính `Σ area`, hình bao và diện tích bao, tập góc lẻ cuối cùng. Điều kiện nào hỏng? Những điểm thừa trong tập góc nằm ở đâu trên hình vẽ?

**Bài 3 — Điền TODO.** Hoàn thiện `exercise_is_rectangle_cover` (cách 3) ở cell dưới.

**Bài 4 — Biến thể 1 chiều.** Viết `exact_cover_1d(segments)`: các đoạn `[l, r]` (`l < r`) có phủ chính xác đoạn `[min l, max r]` (không hở, không chồng; chạm đầu mút thì được) hay không. Yêu cầu `O(n)`, dùng đúng mẹo "độ dài + đầu mút bật/tắt". Thử nghĩ: vì sao ở 1 chiều cũng cần **cả hai** điều kiện?

**Bài 5 — Tự viết lại.** Viết `count_overlapping_pairs(rects)` trả về **số cặp** hình chồng nhau (diện tích chung dương), tự viết cả hàm kiểm tra chồng của hai hình.

**Bài 6 — Mini challenge: săn phản ví dụ.** Viết `find_counterexample(check, trials, seed)` sinh ngẫu nhiên các bộ 2–4 hình nhỏ (tọa độ `0..3`) và trả về bộ **đầu tiên** mà `check(rects) != is_cover_cells(rects)` (hoặc `None`). Dùng nó để tự tìm phản ví dụ cho `wrong3_area_only` và `wrong1_no_area`, rồi `draw` chúng ra.

In [33]:
# Bài 3 — TODO: điền vào chỗ trống
def exercise_is_rectangle_cover(rects):
    # TODO 1: khởi tạo hình bao X, Y, A, B từ rects[0]; total = 0; corner_set = set()
    # TODO 2: duyệt từng hình (x, y, a, b): cập nhật hình bao bằng min/max
    # TODO 3: cộng diện tích (a - x) * (b - y) vào total
    # TODO 4: bật/tắt 4 góc (x,y), (a,y), (x,b), (a,b) trong corner_set
    # TODO 5: trả False nếu total != (A - X) * (B - Y)
    # TODO 6: trả corner_set == {4 góc của hình bao}
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_is_rectangle_cover(ex1) is True
# assert exercise_is_rectangle_cover(ex2) is False
# assert exercise_is_rectangle_cover(cx_corner) is False
# assert exercise_is_rectangle_cover(cx_area) is False

In [34]:
# Bài 4 — TODO: biến thể 1 chiều
def exact_cover_1d(segments):
    # TODO: tổng độ dài + tập đầu mút bật/tắt phải còn đúng {min l, max r}
    raise NotImplementedError

# assert exact_cover_1d([[1, 3], [3, 4]]) is True
# assert exact_cover_1d([[1, 3], [2, 4]]) is False          # chồng
# assert exact_cover_1d([[1, 2], [3, 4]]) is False          # hở
# assert exact_cover_1d([[1, 2], [1, 2], [1, 2]]) is False  # đầu mút qua, độ dài lộ

In [35]:
# Bài 5 — TODO: đếm số cặp chồng nhau
def exercise_rects_overlap(r1, r2):
    # TODO: chồng trên trục x VÀ chồng trên trục y (dấu < nghiêm ngặt)
    pass

def count_overlapping_pairs(rects):
    # TODO: hai vòng lặp i < j, đếm số cặp exercise_rects_overlap(...) là True
    pass

# assert count_overlapping_pairs(ex1) == 0
# assert count_overlapping_pairs(ex3) == 1
# assert count_overlapping_pairs(cx_corner) == 3

In [36]:
# Bài 6 — TODO: săn phản ví dụ
def find_counterexample(check, trials=20000, seed=0):
    # TODO 1: rng = random.Random(seed)
    # TODO 2: lặp trials lần: sinh 2..4 hình [x, y, a, b] với 0 <= x < a <= 3, 0 <= y < b <= 3
    # TODO 3: nếu check(rects) != is_cover_cells(rects): return rects
    # TODO 4: hết vòng lặp: return None
    pass

# ce = find_counterexample(wrong3_area_only); print(ce); draw(ce)
# ce = find_counterexample(wrong1_no_area);   print(ce); draw(ce)

## 21. Gợi ý theo mức

**Bài 1**
- Mức 1: hình thứ ba `[0,1,2,2]` là "nắp" phủ cả hàng trên.
- Mức 2: sau hình 2, hai điểm chung `(1,0)` và `(1,1)` bị tắt.
- Mức 3: sau hình 3, `(0,1)` và `(2,1)` bị tắt, `(0,2)` và `(2,2)` được bật.

**Bài 2**
- Mức 1: vẽ hình — có một cột hở.
- Mức 2: hai cột trái và phải mỗi cột tự ghép khít thành một hình `1 × 3`.
- Mức 3: tập góc lẻ = 4 góc của cột trái + 4 góc của cột phải.

**Bài 3**
- Mức 1: chính là mục 10.
- Mức 2: `X, Y, A, B = rects[0]` rồi `X = min(X, x)`… cho từng hình.
- Mức 3: với mỗi góc `p`: nếu `p in corner_set` thì `corner_set.remove(p)`, ngược lại `corner_set.add(p)` (hoặc viết gọn `corner_set ^= {p}`).

**Bài 4**
- Mức 1: "góc" của một đoạn là 2 đầu mút; "diện tích" là độ dài `r − l`.
- Mức 2: trong phủ chính xác, mỗi điểm bên trong là đầu mút của đúng 2 đoạn (một kết thúc, một bắt đầu).
- Mức 3: `return total == R − L and ends == {L, R}`, với `ends` là set bật/tắt.

**Bài 5**
- Mức 1: dùng lại công thức `max(trái) < min(phải)` cho cả hai trục.
- Mức 2: `for i in range(n): for j in range(i + 1, n):`.
- Mức 3: `count += exercise_rects_overlap(rects[i], rects[j])` (`True` cộng như `1`).

**Bài 6**
- Mức 1: chỉ cần sinh dữ liệu ngẫu nhiên và so với trọng tài.
- Mức 2: `x = rng.randint(0, 2); a = rng.randint(x + 1, 3)` đảm bảo `x < a`.
- Mức 3: với `wrong1_no_area` phản ví dụ thường là các hình trùng nhau 3 lần; với `wrong3_area_only` là chồng bù hở.

## 22. Đáp án tham khảo

### Bài 1

```text
Hình 1 [0,0,1,1]: +(0,0) +(1,0) +(0,1) +(1,1)  -> {(0,0),(1,0),(0,1),(1,1)}
Hình 2 [1,0,2,1]: -(1,0) +(2,0) -(1,1) +(2,1)  -> {(0,0),(0,1),(2,0),(2,1)}
Hình 3 [0,1,2,2]: -(0,1) -(2,1) +(0,2) +(2,2)  -> {(0,0),(2,0),(0,2),(2,2)}
Tổng diện tích = 1 + 1 + 2 = 4; hình bao [0,0,2,2] diện tích 4.
Tập góc = 4 góc bao, diện tích khớp -> True.
```

### Bài 2

- `Σ area = 2 + 1 + 1 + 2 = 6`; hình bao `[1,1,4,4]`, diện tích `9` → **diện tích hỏng** (thiếu 3 = đúng 3 ô của cột hở `x = 2`).
- Tập góc lẻ cuối: `{(1,1),(2,1),(1,4),(2,4),(3,1),(4,1),(3,4),(4,4)}` — 8 điểm → **điều kiện góc cũng hỏng**. Các điểm thừa `(2,1), (2,4), (3,1), (3,4)` chính là 4 góc của **khe hở** `[2,1,3,4]`.

### Bài 4 — vì sao cũng cần cả hai điều kiện?

`[[1,2],[1,2],[1,2]]`: đầu mút `1` và `2` mỗi cái gặp 3 lần (lẻ) → tập đầu mút qua, nhưng độ dài `3 ≠ 1`. Ngược lại `[[0,2],[0,1],[3,4]]`: độ dài `2 + 1 + 1 = 4 = 4 − 0` nhưng chồng ở `[0,1]` và hở ở `[2,3]` — tập đầu mút `{1, 2, 3, 4}` lộ ra. Đây đúng là phiên bản 1 chiều của `cx_corner` và `cx_area`.

Cell dưới kiểm chứng bài 1 và bài 2 bằng `trace_corners`; các cell tiếp theo là đáp án code bài 3–6.

In [37]:
print("Bài 1:", trace_corners([[0, 0, 1, 1], [1, 0, 2, 1], [0, 1, 2, 2]]))
print()
print("Bài 2:", trace_corners(ex2))

A=[0, 0, 1, 1] S=1 | +(0,0) +(1,0) +(0,1) +(1,1) | |set|=4 | tổng S=1
B=[1, 0, 2, 1] S=1 | -(1,0) +(2,0) -(1,1) +(2,1) | |set|=4 | tổng S=2
C=[0, 1, 2, 2] S=2 | -(0,1) -(2,1) +(0,2) +(2,2) | |set|=4 | tổng S=4
set cuối    : (0,0) (0,2) (2,0) (2,2)
4 góc bao   : (0,0) (0,2) (2,0) (2,2)
diện tích: 4 vs 4 -> True | góc khớp -> True
Bài 1: True

A=[1, 1, 2, 3] S=2 | +(1,1) +(2,1) +(1,3) +(2,3) | |set|=4 | tổng S=2
B=[1, 3, 2, 4] S=1 | -(1,3) -(2,3) +(1,4) +(2,4) | |set|=4 | tổng S=3
C=[3, 1, 4, 2] S=1 | +(3,1) +(4,1) +(3,2) +(4,2) | |set|=8 | tổng S=4
D=[3, 2, 4, 4] S=2 | -(3,2) -(4,2) +(3,4) +(4,4) | |set|=8 | tổng S=6
set cuối    : (1,1) (1,4) (2,1) (2,4) (3,1) (3,4) (4,1) (4,4)
4 góc bao   : (1,1) (1,4) (4,1) (4,4)
diện tích: 6 vs 9 -> False | góc khớp -> False
Bài 2: False


In [38]:
# Đáp án bài 3
def exercise_is_rectangle_cover(rects):
    X, Y, A, B = rects[0]
    total = 0
    corner_set = set()
    for x, y, a, b in rects:
        X, Y = min(X, x), min(Y, y)
        A, B = max(A, a), max(B, b)
        total += (a - x) * (b - y)
        for p in ((x, y), (a, y), (x, b), (a, b)):
            if p in corner_set:
                corner_set.remove(p)
            else:
                corner_set.add(p)
    if total != (A - X) * (B - Y):
        return False
    return corner_set == {(X, Y), (A, Y), (X, B), (A, B)}

assert exercise_is_rectangle_cover(ex1) is True
assert exercise_is_rectangle_cover(ex2) is False
assert exercise_is_rectangle_cover(cx_corner) is False
assert exercise_is_rectangle_cover(cx_area) is False
assert exercise_is_rectangle_cover(pinwheel) is True
print("Bài 3 PASS")

Bài 3 PASS


In [39]:
# Đáp án bài 4
def exact_cover_1d(segments):
    L = min(l for l, r in segments)
    R = max(r for l, r in segments)
    total = 0
    ends = set()
    for l, r in segments:
        total += r - l
        for p in (l, r):
            if p in ends:
                ends.remove(p)
            else:
                ends.add(p)
    return total == R - L and ends == {L, R}

def exact_cover_1d_brute(segments):
    painted = set()
    for l, r in segments:
        for i in range(l, r):
            if i in painted:
                return False
            painted.add(i)
    return len(painted) == max(r for _, r in segments) - min(l for l, _ in segments)

assert exact_cover_1d([[1, 3], [3, 4]]) is True
assert exact_cover_1d([[1, 3], [2, 4]]) is False
assert exact_cover_1d([[1, 2], [3, 4]]) is False
assert exact_cover_1d([[1, 2], [1, 2], [1, 2]]) is False
assert exact_cover_1d([[1, 4], [1, 2], [2, 4]]) is False      # chồng 2 lớp: đầu mút tắt hết
rng = random.Random(4)
for _ in range(3000):
    segs = []
    for _ in range(rng.randint(1, 5)):
        l = rng.randint(0, 6)
        segs.append([l, l + rng.randint(1, 3)])
    assert exact_cover_1d(segs) == exact_cover_1d_brute(segs), segs
print("Bài 4 PASS (kể cả 3000 test ngẫu nhiên so với brute force)")

Bài 4 PASS (kể cả 3000 test ngẫu nhiên so với brute force)


In [40]:
# Đáp án bài 5
def exercise_rects_overlap(r1, r2):
    return (max(r1[0], r2[0]) < min(r1[2], r2[2]) and
            max(r1[1], r2[1]) < min(r1[3], r2[3]))

def count_overlapping_pairs(rects):
    n = len(rects)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            count += exercise_rects_overlap(rects[i], rects[j])
    return count

assert count_overlapping_pairs(ex1) == 0
assert count_overlapping_pairs(ex3) == 1
assert count_overlapping_pairs(cx_corner) == 3
assert count_overlapping_pairs(cx_area) == 1
print("Bài 5 PASS; số cặp chồng của cx_bbox =", count_overlapping_pairs(cx_bbox))

Bài 5 PASS; số cặp chồng của cx_bbox = 1


In [41]:
# Đáp án bài 6
def find_counterexample(check, trials=20000, seed=0):
    rng = random.Random(seed)
    for _ in range(trials):
        rects = []
        for _ in range(rng.randint(2, 4)):
            x = rng.randint(0, 2); a = rng.randint(x + 1, 3)
            y = rng.randint(0, 2); b = rng.randint(y + 1, 3)
            rects.append([x, y, a, b])
        if check(rects) != is_cover_cells(rects):
            return rects
    return None

for wrong in [wrong3_area_only, wrong1_no_area]:
    ce = find_counterexample(wrong)
    assert ce is not None and wrong(ce) != is_cover_cells(ce)
    print(f"{wrong.__name__}: phản ví dụ {ce} (hàm sai trả {wrong(ce)})")
    draw(ce)
assert find_counterexample(Solution().isRectangleCover) is None
print("Lời giải chính: không tìm thấy phản ví dụ nào trong 20000 lần thử")

wrong3_area_only: phản ví dụ [[1, 2, 2, 3], [2, 2, 3, 3], [1, 2, 2, 3], [1, 1, 2, 2]] (hàm sai trả True)
  y= 2 | 2 B
  y= 1 | D .
         +----
      x =   1 2
wrong1_no_area: phản ví dụ [[1, 1, 2, 2], [1, 1, 2, 2], [1, 1, 2, 2]] (hàm sai trả True)
  y= 1 | 3
         +--
      x =   1


Lời giải chính: không tìm thấy phản ví dụ nào trong 20000 lần thử


## 23. Template ngắn dễ nhớ

```python
def isRectangleCover(rectangles):
    X, Y, A, B = rectangles[0]
    total, corners = 0, set()
    for x, y, a, b in rectangles:
        X, Y, A, B = min(X, x), min(Y, y), max(A, a), max(B, b)   # 1. hình bao
        total += (a - x) * (b - y)                                # 2. diện tích
        for p in ((x, y), (a, y), (x, b), (a, b)):                # 3. bật/tắt góc
            corners ^= {p}
    return (total == (A - X) * (B - Y) and                        # diện tích khớp
            corners == {(X, Y), (A, Y), (X, B), (A, B)})          # chỉ còn 4 góc bao
```

Câu thần chú: **"Bao – Diện tích – Bật tắt góc – So cả tập."**

- Diện tích bắt **sai về lượng** (thiếu hoặc thừa).
- Góc lẻ bắt **sai về hình dạng** (chồng bù hở, hình chữ L, mảnh rời…).
- Chỉ khi cả hai cùng đúng mới là phủ hoàn hảo.

## 24. Checklist trước khi submit

- [ ] Hình bao lấy `min` của `x, y` và `max` của `a, b` (không nhầm `a` với `x`).
- [ ] Diện tích từng hình là `(a − x) · (b − y)`; so `total` với `(A − X) · (B − Y)`.
- [ ] Góc lưu dạng **tuple** `(x, y)`, không phải list.
- [ ] Mỗi góc được **bật/tắt** (thêm nếu chưa có, xóa nếu đã có) — không phải chỉ thêm.
- [ ] So **cả tập** góc `== {4 góc bao}`, không chỉ kiểm "4 góc có trong set".
- [ ] Có **cả hai** điều kiện (nhớ hai phản ví dụ: 3 hình trùng nhau; chồng bù hở).
- [ ] Đã thử: 3 ví dụ của đề, `n = 1`, hai hình trùng, chỉ chạm góc, hình chữ L, hình chong chóng.
- [ ] Nếu viết C++/Java: diện tích dùng kiểu 64-bit.
- [ ] Độ phức tạp `O(n)` thời gian, `O(n)` bộ nhớ — chạy tốt với `n = 2·10^4`.